In [1]:
from pathlib import Path
import json
import sys
from datetime import datetime

import pandas as pd

# Make sure the project root is available
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name != "ecohome_solution":
    PROJECT_ROOT = Path.cwd()

sys.path.insert(0, str(PROJECT_ROOT))

from agent import EcoHomeAgent

print("EcoHome evaluation notebook loaded")
print("Project root:", PROJECT_ROOT)

EcoHome evaluation notebook loaded
Project root: c:\Users\praveen.vanaparthy\Downloads\New folder\ecohome_solution


In [2]:
ECOHOME_SYSTEM_PROMPT = """
You are EcoHome Energy Advisor, an AI-powered household energy optimization assistant.

ROLE:
Help homeowners understand electricity usage and provide practical,
data-driven recommendations to reduce energy consumption and electricity cost.

YOUR PROCESS:
1. Understand the user's question and household context.
2. Retrieve relevant energy-efficiency knowledge using RAG when appropriate.
3. Check historical energy usage when device usage or consumption patterns are relevant.
4. Check electricity prices when time-of-use or peak/off-peak decisions matter.
5. Check weather when weather conditions can affect HVAC or solar recommendations.
6. Check solar generation when solar usage or generation is relevant.
7. Calculate potential savings when enough numerical information is available.
8. Combine the available evidence before making a recommendation.
9. Clearly explain the recommendation and the reasoning behind it.
10. Never invent energy measurements, prices, weather information, or savings.

CAPABILITIES:
- Energy usage analysis
- Device-level energy analysis
- Solar generation analysis
- Weather-aware recommendations
- Electricity price optimization
- Energy-efficiency knowledge retrieval
- Savings estimation
- EV charging optimization
- HVAC optimization
- Smart-home energy management
- Energy-storage recommendations

RECOMMENDATION GUIDELINES:
- Prefer practical actions that a homeowner can implement.
- Prioritize actions with measurable cost or energy benefits.
- Use historical data when available.
- Consider peak and off-peak electricity pricing.
- Consider solar generation when available.
- Explain assumptions and limitations.
- Identify simulated/mock external data when applicable.
- Do not claim that an action will save a specific amount unless the available
  data supports the calculation.

EXAMPLE QUESTIONS:
1. How can I reduce my electricity cost during peak hours?
2. Which appliance is consuming the most energy?
3. When should I charge my EV?
4. How can I optimize my HVAC usage?
5. How much could I save by shifting usage to off-peak hours?
6. How can I make better use of my solar generation?
7. Should I consider battery storage?
8. What can I do to reduce my household energy consumption?

FINAL RESPONSE:
Provide a concise, clear and actionable recommendation.
Include relevant numerical findings when available.
"""
print("ECOHOME_SYSTEM_PROMPT created")
print("Prompt length:", len(ECOHOME_SYSTEM_PROMPT))

ECOHOME_SYSTEM_PROMPT created
Prompt length: 2362


In [5]:
agent = EcoHomeAgent(
    instructions=ECOHOME_SYSTEM_PROMPT,
    model="command-a-03-2025"
)

print("EcoHome agent initialized successfully")
print("Model:", agent.model_name)
print("Tool count:", 6)

EcoHome agent initialized successfully
Model: command-a-03-2025
Tool count: 6


In [6]:
test_cases = [
    {
        "id": "TC01",
        "question": "How can I reduce my electricity cost during peak hours?",
        "expected_tools": ["search_energy_tips", "get_electricity_prices"],
    },
    {
        "id": "TC02",
        "question": "Which devices have used the most energy in my home during the last 7 days?",
        "expected_tools": ["get_energy_usage"],
    },
    {
        "id": "TC03",
        "question": "How should I optimize my EV charging schedule?",
        "expected_tools": ["search_energy_tips", "get_electricity_prices"],
    },
    {
        "id": "TC04",
        "question": "How can I reduce HVAC electricity consumption?",
        "expected_tools": ["search_energy_tips", "get_weather_forecast"],
    },
    {
        "id": "TC05",
        "question": "How much solar energy has been generated recently?",
        "expected_tools": ["get_solar_generation"],
    },
    {
        "id": "TC06",
        "question": "What are the best ways to use solar generation to reduce grid electricity usage?",
        "expected_tools": ["get_solar_generation", "search_energy_tips"],
    },
    {
        "id": "TC07",
        "question": "How can I reduce energy consumption during hot weather?",
        "expected_tools": ["get_weather_forecast", "search_energy_tips"],
    },
    {
        "id": "TC08",
        "question": "Would shifting electricity usage from peak hours to off-peak hours save money?",
        "expected_tools": ["get_electricity_prices", "calculate_savings"],
    },
    {
        "id": "TC09",
        "question": "Should I consider battery storage for my solar system?",
        "expected_tools": ["search_energy_tips", "get_solar_generation"],
    },
    {
        "id": "TC10",
        "question": "Give me recommendations to reduce my overall household energy consumption.",
        "expected_tools": ["get_energy_usage", "search_energy_tips"],
    },
    {
        "id": "TC11",
        "question": "How can smart-home automation help reduce my electricity usage?",
        "expected_tools": ["search_energy_tips"],
    },
    {
        "id": "TC12",
        "question": "What can I do to optimize my pool pump and washing machine usage?",
        "expected_tools": ["get_energy_usage", "search_energy_tips"],
    },
]

print("Total test cases:", len(test_cases))

for test in test_cases:
    print(test["id"], "-", test["question"])

Total test cases: 12
TC01 - How can I reduce my electricity cost during peak hours?
TC02 - Which devices have used the most energy in my home during the last 7 days?
TC03 - How should I optimize my EV charging schedule?
TC04 - How can I reduce HVAC electricity consumption?
TC05 - How much solar energy has been generated recently?
TC06 - What are the best ways to use solar generation to reduce grid electricity usage?
TC07 - How can I reduce energy consumption during hot weather?
TC08 - Would shifting electricity usage from peak hours to off-peak hours save money?
TC09 - Should I consider battery storage for my solar system?
TC10 - Give me recommendations to reduce my overall household energy consumption.
TC11 - How can smart-home automation help reduce my electricity usage?
TC12 - What can I do to optimize my pool pump and washing machine usage?


In [7]:
test_results = []

for test in test_cases:
    print(f"\nRunning {test['id']}...")

    result = agent.invoke(test["question"])

    test_results.append({
        "test_id": test["id"],
        "question": test["question"],
        "response": result["response"],
        "used_tools": result["used_tools"],
        "expected_tools": test["expected_tools"],
        "success": result["success"],
        "errors": result["errors"],
    })

    print("Success:", result["success"])
    print("Tools:", result["used_tools"])

print("\nAll test cases completed.")
print("Total results:", len(test_results))


Running TC01...
Success: True
Tools: ['search_energy_tips']

Running TC02...
Success: True
Tools: ['get_energy_usage']

Running TC03...
Success: True
Tools: ['get_energy_usage', 'get_electricity_prices']

Running TC04...
Success: True
Tools: ['search_energy_tips']

Running TC05...
Success: True
Tools: ['get_solar_generation']

Running TC06...
Success: True
Tools: ['search_energy_tips']

Running TC07...
Success: True
Tools: ['search_energy_tips']

Running TC08...
Success: True
Tools: ['search_energy_tips']

Running TC09...
Success: True
Tools: ['search_energy_tips']

Running TC10...
Success: True
Tools: ['search_energy_tips']

Running TC11...
Success: True
Tools: ['search_energy_tips']

Running TC12...
Success: True
Tools: ['search_energy_tips']

All test cases completed.
Total results: 12


In [8]:
results_df = pd.DataFrame(test_results)

display(
    results_df[
        ["test_id", "success", "used_tools", "question"]
    ]
)

print("\nSuccessful tests:",
      results_df["success"].sum(),
      "/",
      len(results_df))

,test_id,success,used_tools,question
0,TC01,True,[search_energy_tips],How can I reduce my electricity cost during pe...
1,TC02,True,[get_energy_usage],Which devices have used the most energy in my ...
2,TC03,True,"[get_energy_usage, get_electricity_prices]",How should I optimize my EV charging schedule?
3,TC04,True,[search_energy_tips],How can I reduce HVAC electricity consumption?
4,TC05,True,[get_solar_generation],How much solar energy has been generated recen...
5,TC06,True,[search_energy_tips],What are the best ways to use solar generation...
6,TC07,True,[search_energy_tips],How can I reduce energy consumption during hot...
7,TC08,True,[search_energy_tips],Would shifting electricity usage from peak hou...
8,TC09,True,[search_energy_tips],Should I consider battery storage for my solar...
9,TC10,True,[search_energy_tips],Give me recommendations to reduce my overall h...



Successful tests: 12 / 12


In [9]:
def evaluate_response(response, question):
    """
    Evaluate the quality of an EcoHome agent response.

    Scores:
        Accuracy
        Relevance
        Completeness
        Usefulness

    Each metric is scored from 1 to 5.
    """

    response_text = str(response).strip().lower()
    question_text = str(question).strip().lower()

    if not response_text:
        return {
            "Accuracy": 1,
            "Relevance": 1,
            "Completeness": 1,
            "Usefulness": 1,
            "Feedback": "No response was generated.",
        }

    # Basic deterministic evaluation suitable for repeatable testing.
    accuracy = 5

    # Penalize obvious unsupported/error responses.
    if "could not" in response_text or "error" in response_text:
        accuracy = 2

    relevance = 5

    # Check whether the response addresses important words
    # from the user's question.
    question_keywords = [
        word for word in question_text.split()
        if len(word) > 4
    ]

    matching_keywords = sum(
        1 for word in question_keywords
        if word in response_text
    )

    if question_keywords:
        match_ratio = matching_keywords / len(question_keywords)

        if match_ratio < 0.20:
            relevance = 2
        elif match_ratio < 0.40:
            relevance = 3
        elif match_ratio < 0.60:
            relevance = 4

    completeness = 5

    important_terms = [
        "recommend",
        "energy",
        "electricity",
        "cost",
    ]

    missing_terms = [
        term for term in important_terms
        if term not in response_text
    ]

    if len(missing_terms) >= 3:
        completeness = 2
    elif len(missing_terms) == 2:
        completeness = 3
    elif len(missing_terms) == 1:
        completeness = 4

    usefulness = 5

    action_terms = [
        "shift",
        "schedule",
        "reduce",
        "optimize",
        "charge",
        "use",
        "avoid",
        "consider",
        "recommend",
    ]

    action_count = sum(
        1 for term in action_terms
        if term in response_text
    )

    if action_count == 0:
        usefulness = 2
    elif action_count <= 2:
        usefulness = 3
    elif action_count <= 4:
        usefulness = 4

    feedback_parts = []

    if accuracy >= 4:
        feedback_parts.append("Response appears consistent with the available data.")
    else:
        feedback_parts.append("Review factual accuracy and tool-derived evidence.")

    if relevance >= 4:
        feedback_parts.append("Response directly addresses the question.")
    else:
        feedback_parts.append("Improve alignment with the user's question.")

    if completeness >= 4:
        feedback_parts.append("Key recommendation information is present.")
    else:
        feedback_parts.append("Add missing energy, cost, or recommendation details.")

    if usefulness >= 4:
        feedback_parts.append("Response contains practical actions.")
    else:
        feedback_parts.append("Provide more actionable steps.")

    return {
        "Accuracy": accuracy,
        "Relevance": relevance,
        "Completeness": completeness,
        "Usefulness": usefulness,
        "Feedback": " ".join(feedback_parts),
    }


print("evaluate_response() created successfully")

evaluate_response() created successfully


In [10]:
def evaluate_tool_usage(used_tools, expected_tools):
    """
    Evaluate whether the agent selected appropriate and complete tools.

    Scores:
        Tool Appropriateness: 1-5
        Tool Completeness: 1-5
    """

    used = set(used_tools or [])
    expected = set(expected_tools or [])

    if not used:
        return {
            "Tool Appropriateness": 1,
            "Tool Completeness": 1,
            "Feedback": "No tools were used.",
        }

    relevant_used = used.intersection(expected)

    # Appropriateness:
    # Higher when selected tools are relevant to the expected tools.
    if len(relevant_used) == len(used):
        appropriateness = 5
    elif len(relevant_used) >= max(1, len(used) // 2):
        appropriateness = 4
    else:
        appropriateness = 2

    # Completeness:
    # Measures how many expected tools were actually selected.
    if not expected:
        completeness = 5
    else:
        coverage = len(relevant_used) / len(expected)

        if coverage >= 1.0:
            completeness = 5
        elif coverage >= 0.75:
            completeness = 4
        elif coverage >= 0.50:
            completeness = 3
        elif coverage > 0:
            completeness = 2
        else:
            completeness = 1

    if completeness >= 4 and appropriateness >= 4:
        feedback = "Tool selection was appropriate and sufficiently complete."
    elif appropriateness >= 4:
        feedback = "Selected tools were relevant, but some expected tools were not used."
    elif completeness >= 4:
        feedback = "Most expected tools were used, but some tool selections may be unnecessary."
    else:
        feedback = "Tool selection should be reviewed for relevance and completeness."

    return {
        "Tool Appropriateness": appropriateness,
        "Tool Completeness": completeness,
        "Feedback": feedback,
    }


print("evaluate_tool_usage() created successfully")

evaluate_tool_usage() created successfully


In [11]:
evaluation_results = []

for test in test_results:
    response_eval = evaluate_response(
        test["response"],
        test["question"]
    )

    tool_eval = evaluate_tool_usage(
        test["used_tools"],
        test["expected_tools"]
    )

    evaluation_results.append({
        "test_id": test["test_id"],
        "question": test["question"],
        "success": test["success"],
        "used_tools": ", ".join(test["used_tools"]),
        "expected_tools": ", ".join(test["expected_tools"]),

        "Accuracy": response_eval["Accuracy"],
        "Relevance": response_eval["Relevance"],
        "Completeness": response_eval["Completeness"],
        "Usefulness": response_eval["Usefulness"],

        "Tool Appropriateness": tool_eval["Tool Appropriateness"],
        "Tool Completeness": tool_eval["Tool Completeness"],

        "Response Feedback": response_eval["Feedback"],
        "Tool Feedback": tool_eval["Feedback"],
    })

evaluation_df = pd.DataFrame(evaluation_results)

print("Evaluation completed.")
print("Evaluated test cases:", len(evaluation_df))

display(
    evaluation_df[
        [
            "test_id",
            "Accuracy",
            "Relevance",
            "Completeness",
            "Usefulness",
            "Tool Appropriateness",
            "Tool Completeness",
        ]
    ]
)

Evaluation completed.
Evaluated test cases: 12


,test_id,Accuracy,Relevance,Completeness,Usefulness,Tool Appropriateness,Tool Completeness
0,TC01,5,5,4,5,5,3
1,TC02,5,4,3,4,5,5
2,TC03,5,4,4,5,4,3
3,TC04,5,5,5,5,5,3
4,TC05,5,5,2,3,5,5
5,TC06,5,5,4,5,5,3
6,TC07,5,5,4,4,5,3
7,TC08,5,5,3,4,2,1
8,TC09,5,5,5,5,5,3
9,TC10,5,5,4,4,5,3


In [12]:
def generate_evaluation_report(evaluation_df):
    """
    Generate an overall EcoHome agent evaluation report.
    """

    response_metrics = [
        "Accuracy",
        "Relevance",
        "Completeness",
        "Usefulness",
    ]

    tool_metrics = [
        "Tool Appropriateness",
        "Tool Completeness",
    ]

    response_scores = {
        metric: round(evaluation_df[metric].mean(), 2)
        for metric in response_metrics
    }

    tool_scores = {
        metric: round(evaluation_df[metric].mean(), 2)
        for metric in tool_metrics
    }

    overall_score = round(
        sum(response_scores.values()) +
        sum(tool_scores.values()),
        2
    )

    max_score = 5 * (
        len(response_metrics) + len(tool_metrics)
    )

    overall_percentage = round(
        (overall_score / max_score) * 100,
        2
    )

    successful_tests = int(
        evaluation_df["success"].sum()
    )

    total_tests = len(evaluation_df)

    # Strengths
    strengths = []

    for metric, score in {
        **response_scores,
        **tool_scores,
    }.items():
        if score >= 4:
            strengths.append(
                f"{metric} scored {score}/5."
            )

    if not strengths:
        strengths.append(
            "The agent completed the evaluation workflow."
        )

    # Weaknesses
    weaknesses = []

    for metric, score in {
        **response_scores,
        **tool_scores,
    }.items():
        if score < 4:
            weaknesses.append(
                f"{metric} could be improved ({score}/5)."
            )

    if not weaknesses:
        weaknesses.append(
            "No major weaknesses were identified by the deterministic evaluation."
        )

    recommendations = [
        "Improve tool-selection rules for questions requiring multiple data sources.",
        "Use historical usage and pricing together when numerical optimization is requested.",
        "Use savings calculations whenever sufficient numerical inputs are available.",
        "Add richer personalization using household/device characteristics.",
        "Consider replacing mock weather and pricing services with production APIs.",
        "Add visualizations for energy usage, solar generation, and cost trends.",
    ]

    report = {
        "total_tests": total_tests,
        "successful_tests": successful_tests,
        "success_rate_percent": round(
            successful_tests / total_tests * 100,
            2
        ) if total_tests else 0,

        "response_scores": response_scores,
        "tool_scores": tool_scores,

        "overall_score": overall_score,
        "maximum_score": max_score,
        "overall_percentage": overall_percentage,

        "strengths": strengths,
        "weaknesses": weaknesses,
        "recommendations": recommendations,
    }

    return report


evaluation_report = generate_evaluation_report(
    evaluation_df
)

print("Evaluation Report")
print("=" * 60)

print(
    f"Tests: {evaluation_report['successful_tests']}/"
    f"{evaluation_report['total_tests']}"
)

print(
    f"Success Rate: "
    f"{evaluation_report['success_rate_percent']}%"
)

print(
    f"Overall Score: "
    f"{evaluation_report['overall_score']}/"
    f"{evaluation_report['maximum_score']}"
)

print(
    f"Overall Percentage: "
    f"{evaluation_report['overall_percentage']}%"
)

print("\nResponse Scores:")
for metric, score in evaluation_report["response_scores"].items():
    print(f"- {metric}: {score}/5")

print("\nTool Scores:")
for metric, score in evaluation_report["tool_scores"].items():
    print(f"- {metric}: {score}/5")

print("\nStrengths:")
for item in evaluation_report["strengths"]:
    print("-", item)

print("\nWeaknesses:")
for item in evaluation_report["weaknesses"]:
    print("-", item)

print("\nRecommendations:")
for item in evaluation_report["recommendations"]:
    print("-", item)

Evaluation Report
Tests: 12/12
Success Rate: 100.0%
Overall Score: 26.08/30
Overall Percentage: 86.93%

Response Scores:
- Accuracy: 5.0/5
- Relevance: 4.83/5
- Completeness: 3.83/5
- Usefulness: 4.42/5

Tool Scores:
- Tool Appropriateness: 4.67/5
- Tool Completeness: 3.33/5

Strengths:
- Accuracy scored 5.0/5.
- Relevance scored 4.83/5.
- Usefulness scored 4.42/5.
- Tool Appropriateness scored 4.67/5.

Weaknesses:
- Completeness could be improved (3.83/5).
- Tool Completeness could be improved (3.33/5).

Recommendations:
- Improve tool-selection rules for questions requiring multiple data sources.
- Use historical usage and pricing together when numerical optimization is requested.
- Use savings calculations whenever sufficient numerical inputs are available.
- Add richer personalization using household/device characteristics.
- Consider replacing mock weather and pricing services with production APIs.
- Add visualizations for energy usage, solar generation, and cost trends.


In [13]:
# Save detailed test results
results_df.to_csv(
    PROJECT_ROOT / "evaluation_test_results.csv",
    index=False
)

# Save evaluation scores
evaluation_df.to_csv(
    PROJECT_ROOT / "evaluation_scores.csv",
    index=False
)

# Save the complete evaluation report
with open(
    PROJECT_ROOT / "evaluation_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        evaluation_report,
        f,
        indent=2
    )

print("Evaluation artifacts saved successfully.")

print("\nCreated:")
print("- evaluation_test_results.csv")
print("- evaluation_scores.csv")
print("- evaluation_report.json")

Evaluation artifacts saved successfully.

Created:
- evaluation_test_results.csv
- evaluation_scores.csv
- evaluation_report.json
